# Datas e fusos com Python

Use datetime, timedelta e zoneinfo. Nas datas usadas aqui, São Paulo está em UTC−3 e Cuiabá em UTC−4. Use os nomes das zonas para aplicar as regras do fuso, em vez de subtrair horas manualmente.

Use este notebook antes da versão com pandas. Não precisa instalar pacotes: usamos apenas Python e sua biblioteca padrão.

**Como estudar:** execute as demos de cima para baixo. Antes de cada exercício, escreva entradas, saída esperada e passos. Preencha a célula vazia, execute e compare com exemplos. Consulte o gabarito só depois da tentativa.

Uma função precisa ser definida antes de ser chamada. Se aparecer `NameError`, confira o nome e execute novamente a célula da definição. Ao reiniciar o kernel, execute as definições outra vez.

## Demo — Texto vira datetime

strptime lê um texto segundo um formato. %d é dia, %m é mês e %Y é ano com quatro dígitos.

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
data = datetime.strptime("05/10/2026", "%d/%m/%Y")
print(data.strftime("%Y-%m-%d"))
print(data + timedelta(days=7))

## Demo — UTC vira horário local

fromisoformat interpreta o deslocamento +00:00. astimezone preserva o instante e muda a representação local. Uma data sem fuso não informa qual horário local representa.

In [ ]:
instante = datetime.fromisoformat("2026-10-05T13:00:00+00:00")
print(instante.astimezone(ZoneInfo("America/Sao_Paulo")))
print(instante.astimezone(ZoneInfo("America/Cuiaba")))

## Demo — Repetição vira função

Contrato: entrada ISO com deslocamento explícito; saída datetime no destino. Recusamos datas sem fuso, pois não sabemos se representam UTC.

In [ ]:
def converter_fuso(texto, destino):
    momento = datetime.fromisoformat(texto)
    if momento.tzinfo is None:
        raise ValueError("Informe o deslocamento do fuso na entrada.")
    return momento.astimezone(ZoneInfo(destino))

print(converter_fuso("2026-10-05T13:00:00+00:00", "America/Cuiaba"))

## Exercício 1

Converta os três textos para datetime. Primeiro faça três chamadas; depois crie `ler_data(texto)` e use um for. Compare os resultados.

In [ ]:
textos = ["05/10/2026", "06/10/2026", "07/10/2026"]

In [ ]:
# Escreva sua solução aqui.


## Exercício 2

Crie `somar_dias(texto, quantidade)` usando ler_data. Retorne uma string no formato dia/mês/ano. Teste virada de mês e ano.

In [ ]:
# Escreva sua solução aqui.


## Exercício 3

Converta cada horário UTC para São Paulo e Cuiabá chamando converter_fuso. Guarde os resultados em duas listas. Qual é a diferença de horário local?

In [ ]:
eventos = ["2026-10-05T13:00:00+00:00", "2026-10-05T18:30:00+00:00"]

In [ ]:
# Escreva sua solução aqui.


## Exercício 4

Converta este instante para as duas cidades. Confira a data, não apenas a hora. Explique por que a data local muda.

In [ ]:
madrugada = "2026-10-06T02:30:00+00:00"

In [ ]:
# Escreva sua solução aqui.


## Exercício 5

Crie `em_expediente(horario, inicio=8, fim=18)`. Receba um datetime já convertido para a cidade e retorne bool. Considere início incluído e fim excluído. Teste 07:59, 08:00 e 18:00.

In [ ]:
# Escreva sua solução aqui.


## Exercício 6

Crie `resumir_evento(texto, cidade)` que chama converter_fuso e em_expediente, retornando um dicionário com horário formatado e indicador de expediente. Teste também a rejeição de uma entrada sem fuso.

In [ ]:
# Escreva sua solução aqui.


## Gabaritos comentados

Pare aqui se ainda estiver praticando. Compare as regras e os resultados, não apenas a aparência do código.

### Solução 1

In [ ]:
antes = [datetime.strptime(t, "%d/%m/%Y") for t in textos]
def ler_data(texto):
    return datetime.strptime(texto, "%d/%m/%Y")
depois = []
for texto in textos:
    depois.append(ler_data(texto))
assert antes == depois

### Solução 2

In [ ]:
def somar_dias(texto, quantidade):
    return (ler_data(texto) + timedelta(days=quantidade)).strftime("%d/%m/%Y")
assert somar_dias("31/10/2026", 1) == "01/11/2026"
assert somar_dias("31/12/2026", 1) == "01/01/2027"

### Solução 3

In [ ]:
sp = [converter_fuso(t, "America/Sao_Paulo") for t in eventos]
cuiaba = [converter_fuso(t, "America/Cuiaba") for t in eventos]
assert [d.hour for d in sp] == [10, 15]
assert [d.hour for d in cuiaba] == [9, 14]
# Cuiabá mostra uma hora a menos nessas datas; os instantes são iguais.
assert sp[0] == cuiaba[0]

### Solução 4

In [ ]:
sp_noite = converter_fuso(madrugada, "America/Sao_Paulo")
cuiaba_noite = converter_fuso(madrugada, "America/Cuiaba")
assert sp_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 23:30"
assert cuiaba_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 22:30"
# O mesmo instante ainda cai no dia anterior no horário local.

### Solução 5

In [ ]:
def em_expediente(horario, inicio=8, fim=18):
    return inicio <= horario.hour < fim
for texto, esperado in [("2026-10-05T10:59:00+00:00", False), ("2026-10-05T11:00:00+00:00", True), ("2026-10-05T21:00:00+00:00", False)]:
    assert em_expediente(converter_fuso(texto, "America/Sao_Paulo")) == esperado

### Solução 6

In [ ]:
def resumir_evento(texto, cidade):
    local = converter_fuso(texto, cidade)
    return {"horario": local.strftime("%d/%m/%Y %H:%M"), "expediente": em_expediente(local)}
assert resumir_evento(eventos[0], "America/Cuiaba") == {"horario": "05/10/2026 09:00", "expediente": True}
try:
    converter_fuso("2026-10-05T13:00:00", "America/Sao_Paulo")
except ValueError as erro:
    print(erro)
else:
    raise AssertionError("Era necessário rejeitar a entrada sem fuso.")